# Analyse de la Loi de Weber — Performance Musicale au Piano
## Pipeline méthodologique complet
**César Mignon — L3 MIASHS Parcours Sciences Cognitives — Université de Lille**

Ce notebook reproduit l'intégralité du pipeline décrit dans le TER, dans l'ordre des sections de la méthodologie (3.1 à 3.8).

---

## Section 3.1 — Environnement logiciel et imports

In [3]:
# Bibliothèques principales utilisées dans le projet
!pip install pretty_midi music21 -q
import music21
import pretty_midi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Imports OK")

Imports OK


In [4]:
# Téléchargement automatique des fichiers depuis GitHub
import subprocess

BASE_URL = "https://raw.githubusercontent.com/TON_USER/TON_REPO/main/data"

fichiers = [
    "bach_bwv856.midi", "beethoven_woo80.midi", "chopin_op10n2.midi",
    "bach_bwv856.mxl",  "beethoven_woo80.mxl",  "chopin_op10n2.mxl"
]

for f in fichiers:
    subprocess.run(["wget", "-q", f"{BASE_URL}/{f}", "-O", f"/content/{f}"])
    print(f"✓ {f}")

chemin_midi_bach      = "/content/bach_bwv856.midi"
chemin_midi_beethoven = "/content/beethoven_woo80.midi"
chemin_midi_chopin    = "/content/chopin_op10n2.midi"
xml_bach              = "/content/bach_bwv856.mxl"
xml_beethoven         = "/content/beethoven_woo80.mxl"
xml_chopin            = "/content/chopin_op10n2.mxl"

---
## Section 3.2 — Approche initiale : extraction brute des IOI

La formule de base est :
$$\text{IOI} = T_{n+1} - T_n$$
où $T$ représente l'horodatage de l'attaque de la note en secondes.

Cette approche naïve, exacte pour une mélodie monophonique, est insuffisante pour le piano polyphonique.

In [6]:
def extraire_ioi_bruts(midi_path):
    """Extraction brute des IOI : T_{n+1} - T_n sur tous les événements.
    Produit une distribution inexploitable sur le piano (section 3.2)."""
    pm = pretty_midi.PrettyMIDI(midi_path)
    onsets = []
    for instrument in pm.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                onsets.append(note.start)
    onsets = sorted(onsets)
    ioi_bruts = np.diff(onsets)
    return ioi_bruts

# Visualisation de la distribution brute (inexploitable)
ioi_bruts = extraire_ioi_bruts(chemin_midi_beethoven)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ioi_bruts, bins=100, color='gray', edgecolor='white')
ax.set_title("IOI bruts — Beethoven WoO 80 (avant tout filtrage)")
ax.set_xlabel("IOI (secondes)")
ax.set_ylabel("Fréquence")
plt.tight_layout()
plt.show()
print(f"Nombre d'IOI bruts : {len(ioi_bruts)}")
print(f"Min : {ioi_bruts.min():.4f}s | Max : {ioi_bruts.max():.2f}s")

FileNotFoundError: [Errno 2] No such file or directory: 'chemin/vers/beethoven_woo80.midi'

---
## Section 3.3 — Premier obstacle : gestion de la polyphonie

Un accord de 3 notes produit 3 événements MIDI séparés par 2–15 ms.
Ces micro-intervalles ne relèvent pas du timing intervallaire.

**Ajustement :** filtre de tolérance temporelle à **30 ms**.

In [ ]:
def extraire_ioi_filtres_polyphonie(midi_path, seuil_accord_ms=30):
    """Filtre les micro-IOI < seuil_accord_ms (artefacts d'accord).
    Retourne les IOI après fusion des notes simultanées (section 3.3)."""
    pm = pretty_midi.PrettyMIDI(midi_path)
    onsets = []
    for instrument in pm.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                onsets.append(note.start)
    onsets = sorted(onsets)

    # Filtrage : si IOI < seuil, les deux notes forment un accord
    seuil_s = seuil_accord_ms / 1000.0
    onsets_filtres = [onsets[0]]
    for i in range(1, len(onsets)):
        if onsets[i] - onsets_filtres[-1] >= seuil_s:
            onsets_filtres.append(onsets[i])

    ioi_filtres = np.diff(onsets_filtres)
    return ioi_filtres

ioi_polyphonie = extraire_ioi_filtres_polyphonie(chemin_midi_beethoven)
print(f"IOI après filtre polyphonie (seuil 30ms) : {len(ioi_polyphonie)}")
print(f"Réduction par rapport aux IOI bruts : {len(ioi_bruts) - len(ioi_polyphonie)} IOI supprimés")

---
## Section 3.4 — Deuxième obstacle : valeurs extrêmes et expressivité

Deux catégories de valeurs aberrantes subsistent :
- **Longues** : silences structurels entre variations (> 1,5 s)
- **Courtes** : ornements (trilles, appoggiatures) non filtrés par le seuil à 30 ms

**Ajustement :** élagage (*trimming*) avec une borne supérieure à 1,5 s.

In [ ]:
def elaguer_ioi(ioi, borne_sup=1.5, borne_inf=0.03):
    """Élague les IOI hors de la plage [borne_inf, borne_sup].
    Élimine silences structurels et ornements résiduels (section 3.4)."""
    masque = (ioi >= borne_inf) & (ioi <= borne_sup)
    return ioi[masque]

ioi_propres = elaguer_ioi(ioi_polyphonie, borne_sup=1.5, borne_inf=0.03)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(ioi_polyphonie, bins=80, color='orange', edgecolor='white')
axes[0].set_title("Après filtre polyphonie (avant élagage)")
axes[0].set_xlabel("IOI (secondes)")
axes[1].hist(ioi_propres, bins=80, color='steelblue', edgecolor='white')
axes[1].set_title("Après élagage [0,03s — 1,5s]")
axes[1].set_xlabel("IOI (secondes)")
plt.tight_layout()
plt.show()
print(f"IOI après élagage : {len(ioi_propres)} (sur {len(ioi_polyphonie)})")

---
## Section 3.5 — Troisième obstacle : regroupement par type de note

Pour tester la propriété scalaire ($CV = \sigma/\mu$), il faut isoler
des IOI de **même durée cible théorique**.
On s'appuie sur les partitions MusicXML comme *ground truth*.

In [ ]:
# Correspondance type de note music21 -> nom français
TRADUCTIONS = {
    'whole':   'Ronde',
    'half':    'Blanche',
    'quarter': 'Noire',
    'eighth':  'Croche',
    '16th':    'Double-croche',
    '32nd':    'Triple-croche'
}

---
## Section 3.6 — Quatrième obstacle : alignement partition–performance

### Problème de l'alignement linéaire
La k-ième note de la partition ≠ le k-ième IOI du MIDI :
- Polyphonie résiduelle (ornements non filtrés)
- Silences structurels sans équivalent dans la partition
- Exemple Beethoven : **3 164 notes partition** vs **4 541 IOI MIDI** (décalage = 1 377)

### Solution : Dynamic Time Warping (DTW)
Algorithme implémenté en NumPy pur (pas de bibliothèque externe disponible).

In [ ]:
def dtw_path(seq1, seq2):
    """DTW implémenté en NumPy pur (O(n×m)).
    Retourne le chemin d'alignement optimal : liste de tuples (i, j).
    Les deux séquences doivent être préalablement normalisées."""
    n, m = len(seq1), len(seq2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(seq1[i-1] - seq2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # insertion
                dtw_matrix[i, j-1],    # deletion
                dtw_matrix[i-1, j-1]   # match
            )

    # Reconstruction du chemin par backtracking
    path = []
    i, j = n, m
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        options = {
            (i-1, j-1): dtw_matrix[i-1, j-1],
            (i-1, j):   dtw_matrix[i-1, j],
            (i, j-1):   dtw_matrix[i, j-1],
        }
        i, j = min(options, key=options.get)
    path.reverse()
    return path


def aligner_dtw(df_partition, iois_midi, nom_pianiste):
    """Aligne la partition et le MIDI via DTW sur les IOI.
    Les séquences sont normalisées par leur médiane avant alignement.
    Retourne df_partition enrichi d'une colonne 'ioi_joue'."""
    seq_th  = df_partition['ioi_theorique'].values.astype(float)
    seq_mid = iois_midi.astype(float)

    # Normalisation par la médiane (rend les séquences comparables
    # indépendamment du tempo)
    seq_th_n  = seq_th  / np.median(seq_th)
    seq_mid_n = seq_mid / np.median(seq_mid)

    print(f"  DTW : {len(seq_th_n)} x {len(seq_mid_n)} — calcul en cours...")
    path = dtw_path(seq_th_n, seq_mid_n)
    print(f"  Chemin trouvé : {len(path)} points")

    df_path = pd.DataFrame(path, columns=['idx_partition', 'idx_midi'])
    df_path = df_path.drop_duplicates(subset='idx_partition', keep='last')

    df_result = df_partition.copy().reset_index(drop=True)
    idx_map = df_path.set_index('idx_partition')['idx_midi']
    df_result['ioi_joue'] = df_result.index.map(
        lambda i: iois_midi[int(idx_map[i])] if i in idx_map.index else np.nan
    )
    df_result['pianiste'] = nom_pianiste
    return df_result

---
## Section 3.7 — Cinquième obstacle : structure interne du fichier MusicXML

Trois problèmes identifiés dans les fichiers `.mxl` :
1. **Offsets locaux** : music21 retourne des offsets relatifs à la mesure → correction nécessaire
2. **Fusion des mains** : les deux mains sont dans une seule portée → séparation par registre MIDI (seuil = 60)
3. **Offsets non entiers** : artefacts d'encodage (ornements) → arrondi à 0,05

In [ ]:
def construire_flux_melodique(xml_path, seuil_midi=60):
    """Extrait les notes depuis un fichier MusicXML avec :
    - Offsets absolus (offset_mesure + offset_local)
    - Séparation main droite / main gauche par registre (seuil_midi=60)
    - Gestion des accords (note la plus aiguë conservée)
    Corrige les problèmes décrits en section 3.7."""
    score = music21.converter.parse(xml_path)
    data = []

    for part in score.parts:
        part_name = part.partName or part.id
        for measure in part.getElementsByClass('Measure'):
            mesure_num = measure.number
            offset_mesure = float(measure.offset)  # offset absolu de la mesure

            voices = measure.getElementsByClass('Voice')
            streams = [(f'voix_{v.id}', v) for v in voices] if voices else [('voix_1', measure)]

            for voix_id, stream in streams:
                for element in stream.notesAndRests:
                    if isinstance(element, music21.note.Rest):
                        continue

                    # Offset absolu = offset mesure + offset local
                    offset_abs = offset_mesure + float(element.offset)

                    if isinstance(element, music21.chord.Chord):
                        note = max(element.notes, key=lambda n: n.pitch.midi)
                        dur  = element
                    elif isinstance(element, music21.note.Note):
                        note = element
                        dur  = element
                    else:
                        continue

                    type_note = dur.duration.type
                    type_nom  = TRADUCTIONS.get(type_note, type_note)
                    if dur.duration.dots > 0:
                        type_nom += ' pointée'

                    data.append({
                        'portee':          part_name,
                        'voix':            voix_id,
                        # Séparation main droite/gauche par registre MIDI
                        'main':            'droite' if note.pitch.midi >= seuil_midi else 'gauche',
                        'mesure':          mesure_num,
                        'offset':          offset_abs,
                        'note':            note.nameWithOctave,
                        'midi':            note.pitch.midi,
                        'type_theorique':  type_nom,
                        'duree_theorique': float(dur.duration.quarterLength)
                    })

    df = pd.DataFrame(data)
    df = df.sort_values(by='offset').reset_index(drop=True)
    return df


def construire_flux_final(df_complet, main, seuil_arrondi=0.05):
    """Pour une main donnée :
    - Arrondit les offsets (corrige artefacts MXL, ex: 759.995 → 760.0)
    - Dédoublonne : garde la note la plus aiguë par offset
    - Calcule l'IOI théorique entre attaques successives."""
    df = df_complet[df_complet['main'] == main].copy()

    # Arrondi souple des offsets non entiers
    df['offset'] = (df['offset'] / seuil_arrondi).round() * seuil_arrondi

    # Dédoublonnage : un seul événement par offset (note la plus aiguë)
    df = (df.sort_values(['offset', 'midi'], ascending=[True, False])
            .drop_duplicates(subset='offset', keep='first')
            .reset_index(drop=True))

    # IOI théorique = écart entre attaques successives
    df['ioi_theorique'] = df['offset'].diff().shift(-1)
    df = df[df['ioi_theorique'] > 0].copy()
    return df


def lire_midi_par_main(midi_path, seuil_midi=60):
    """Sépare les IOI du MIDI en deux mains par registre.
    Même critère que pour la partition (seuil MIDI = 60)."""
    pm = pretty_midi.PrettyMIDI(midi_path)
    notes_md, notes_mg = [], []

    for instrument in pm.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            if note.pitch >= seuil_midi:
                notes_md.append(note.start)
            else:
                notes_mg.append(note.start)

    ioi_md = np.diff(sorted(notes_md))
    ioi_mg = np.diff(sorted(notes_mg))
    return ioi_md, ioi_mg

---
## Section 3.8 — Bilan : pipeline complet pour une œuvre

Fonction maître qui enchaîne toutes les étapes 3.2 à 3.7.

In [ ]:
def pipeline_complet(xml_path, midi_path, nom_oeuvre):
    """Pipeline complet décrit dans le TER (sections 3.2 à 3.7) :
    1. Extraction des notes depuis la partition MusicXML
    2. Construction des flux par main (droite/gauche)
    3. Extraction et séparation des IOI MIDI par registre
    4. Alignement DTW partition ↔ MIDI pour chaque main
    5. Retourne un DataFrame unifié avec ioi_theorique et ioi_joue"""

    print(f"\n{'='*60}")
    print(f"Traitement : {nom_oeuvre}")
    print(f"{'='*60}")

    # Étape 1 — Extraction partition
    print("\n[1/4] Extraction de la partition MusicXML...")
    df_complet = construire_flux_melodique(xml_path)
    flux_md = construire_flux_final(df_complet, 'droite')
    flux_mg = construire_flux_final(df_complet, 'gauche')
    print(f"      Partition — MD : {len(flux_md)} notes | MG : {len(flux_mg)} notes")

    # Étape 2 — Lecture MIDI
    print("\n[2/4] Lecture MIDI et séparation par registre...")
    ioi_md, ioi_mg = lire_midi_par_main(midi_path)
    print(f"      MIDI — MD : {len(ioi_md)} IOI | MG : {len(ioi_mg)} IOI")

    # Vérification du décalage (problème décrit en section 3.6)
    decalage_md = len(ioi_md) - len(flux_md)
    decalage_mg = len(ioi_mg) - len(flux_mg)
    print(f"      Décalage MD : {decalage_md:+d} | Décalage MG : {decalage_mg:+d}")
    if abs(decalage_md) > 100:
        print(f"      ⚠️  Décalage important → alignement linéaire impossible, DTW nécessaire")

    # Étape 3 — Alignement DTW
    print("\n[3/4] Alignement DTW...")
    df_md = aligner_dtw(flux_md, ioi_md, f"{nom_oeuvre} MD")
    df_mg = aligner_dtw(flux_mg, ioi_mg, f"{nom_oeuvre} MG")

    # Étape 4 — Fusion et nettoyage
    print("\n[4/4] Fusion et nettoyage...")
    df_all = pd.concat([df_md, df_mg], ignore_index=True)
    df_clean = df_all[
        (df_all['ioi_joue'] > 0.03) &
        (df_all['ioi_joue'] < 1.5)
    ].copy()
    df_clean['oeuvre'] = nom_oeuvre
    print(f"      Notes conservées après nettoyage : {len(df_clean)}")

    return df_clean

---
## Section 4 — Résultats : calcul du CV et test de la propriété scalaire

La **loi de Weber** prédit : $\sigma(T) = w \times T$, soit un CV $= \sigma/\mu$ **constant**.

On applique le pipeline aux trois œuvres puis on calcule le CV par type de note.

In [ ]:
# Application du pipeline aux trois œuvres
# (peut prendre quelques minutes à cause du DTW)

df_bach      = pipeline_complet(xml_bach,      chemin_midi_bach,      "Bach BWV 856")
df_beethoven = pipeline_complet(xml_beethoven, chemin_midi_beethoven, "Beethoven WoO 80")
df_chopin    = pipeline_complet(xml_chopin,    chemin_midi_chopin,    "Chopin Op.10 n°2")

In [ ]:
def calculer_weber(df, min_n=5):
    """Calcule µ, σ et CV = σ/µ par type de note.
    Exclut les catégories avec n < min_n (effectif insuffisant)."""
    weber = (df.groupby('type_theorique')['ioi_joue']
               .agg(n='count', moyenne='mean', ecart_type='std')
               .assign(CV=lambda x: x['ecart_type'] / x['moyenne'])
               .sort_values('moyenne'))
    return weber[weber['n'] >= min_n]

print("=== BACH ===")
print(calculer_weber(df_bach).round(4))

print("\n=== BEETHOVEN ===")
print(calculer_weber(df_beethoven).round(4))

print("\n=== CHOPIN ===")
print(calculer_weber(df_chopin).round(4))

In [ ]:
# Tableau comparatif (reproduit le Tableau 2 du TER)
weber_bach      = calculer_weber(df_bach)
weber_beethoven = calculer_weber(df_beethoven)
weber_chopin    = calculer_weber(df_chopin)

# Catégories communes pour la comparaison
categories = ['Triple-croche', 'Double-croche', 'Croche', 'Noire']

print(f"{'Type de note':<20} {'Bach':>12} {'Beethoven':>12} {'Chopin':>12}")
print(f"{'':20} {'n':>6}{'CV':>6} {'n':>6}{'CV':>6} {'n':>6}{'CV':>6}")
print("-" * 56)
for cat in categories:
    b  = f"{int(weber_bach.loc[cat,'n']):>6}{weber_bach.loc[cat,'CV']:>6.2f}"      if cat in weber_bach.index      else "    —     —"
    be = f"{int(weber_beethoven.loc[cat,'n']):>6}{weber_beethoven.loc[cat,'CV']:>6.2f}" if cat in weber_beethoven.index else "    —     —"
    c  = f"{int(weber_chopin.loc[cat,'n']):>6}{weber_chopin.loc[cat,'CV']:>6.2f}"   if cat in weber_chopin.index   else "    —     —"
    print(f"{cat:<20} {b} {be} {c}")

In [ ]:
# Visualisation : CV en fonction de µ pour les trois œuvres
# Un CV constant = propriété scalaire vérifiée

fig, ax = plt.subplots(figsize=(10, 5))

couleurs = {
    'Bach BWV 856':      ('royalblue',  'o'),
    'Beethoven WoO 80':  ('darkorange', 's'),
    'Chopin Op.10 n°2':  ('green',      '^')
}

for nom, df in [('Bach BWV 856', df_bach),
                ('Beethoven WoO 80', df_beethoven),
                ('Chopin Op.10 n°2', df_chopin)]:
    w = calculer_weber(df)
    couleur, marqueur = couleurs[nom]
    ax.plot(w['moyenne'], w['CV'],
            marker=marqueur, linewidth=2, markersize=8,
            color=couleur, label=nom)
    # Annotations
    for idx, row in w.iterrows():
        ax.annotate(idx, (row['moyenne'], row['CV']),
                    textcoords='offset points', xytext=(5, 5),
                    fontsize=7, color=couleur)

# Ligne CV = 1 (référence Weber parfaite)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5,
           label='CV = 1,0 (Weber parfait)')

ax.set_xlabel("IOI moyen µ (secondes)", fontsize=12)
ax.set_ylabel("Coefficient de variation CV = σ/µ", fontsize=12)
ax.set_title("Test de la propriété scalaire : CV en fonction de µ\n"
             "Un CV constant = loi de Weber vérifiée", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(left=0)
plt.tight_layout()
plt.savefig('cv_comparatif.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Histogrammes IOI par œuvre — distribution des doubles-croches
# Illustre l'anomalie 0,5–1s corrigée par le DTW

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, (nom, df, couleur) in zip(axes, [
    ('Bach BWV 856',      df_bach,      'royalblue'),
    ('Beethoven WoO 80',  df_beethoven, 'darkorange'),
    ('Chopin Op.10 n°2',  df_chopin,    'green')
]):
    dc = df[df['type_theorique'] == 'Double-croche']['ioi_joue']
    if len(dc) > 0:
        ax.hist(dc, bins=50, color=couleur, edgecolor='white', alpha=0.85)
        ax.axvline(dc.median(), color='red', linestyle='--',
                   label=f'Médiane : {dc.median():.3f}s')
        ax.set_title(f"{nom}\nDoubles-croches (n={len(dc)})")
        ax.set_xlabel("IOI joué (secondes)")
        ax.set_ylabel("Fréquence")
        ax.legend(fontsize=8)

plt.suptitle("Distribution des IOI joués — Doubles-croches (après DTW)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('histogrammes_doubles_croches.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Estimation du tempo depuis les doubles-croches
# IOI_absolu = 0,25 / tempo_BPM × 60

for nom, df in [('Bach BWV 856', df_bach),
                ('Beethoven WoO 80', df_beethoven),
                ('Chopin Op.10 n°2', df_chopin)]:
    dc = df[
        (df['type_theorique'] == 'Double-croche') &
        (df['ioi_joue'] > 0.05) &
        (df['ioi_joue'] < 0.5)
    ]['ioi_joue']
    if len(dc) > 0:
        tempo = (0.25 / dc.median()) * 60
        print(f"{nom} — Médiane DC : {dc.median():.3f}s → Tempo estimé : {tempo:.0f} BPM")

---
## Synthèse des résultats (Section 4.1.4)

| Régime | Œuvre(s) | CV | Propriété scalaire |
|--------|----------|----|-------------------|
| **Timing automatique** | Bach, Chopin (DC) | Faible et décroissant | ✗ Absente |
| **Timing cognitif-expressif** | Beethoven (notes courtes) | ~1,0, stable | ✓ Partielle |

**Interprétation** : La loi de Weber s'applique au timing intervallaire, mais le piano expert mobilise en parallèle un système de timing automatique (cérébelleux) qui efface la signature scalaire dans les passages rapides.